# GPT-2 Exploration

This notebook explores a pretrained GPT-2 model and compares
its tokenization, architecture, attention mechanisms, and
text generation with our Tiny GPT poem generator.

In [2]:
!pip install transformers torch

In [3]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

In [4]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
print(tokenizer)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT2Tokenizer(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
})


## 2. Exploring GPT-2 Tokenization

GPT-2 does not process raw text directly. It first converts text into
tokens, and then converts those tokens into numerical token IDs.

We will examine:
- The original text
- The tokens created by GPT-2
- The numerical token IDs
- How the IDs can be converted back into text

In [5]:
prompt = "The moon shines over the hills"

# Convert text into GPT-2 tokens
tokens = tokenizer.tokenize(prompt)

# Convert text into numerical token IDs
token_ids = tokenizer.encode(prompt)

# Convert token IDs back into text
decoded_text = tokenizer.decode(token_ids)

print("Original text:")
print(prompt)

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(token_ids)

print("\nDecoded text:")
print(decoded_text)

Original text:
The moon shines over the hills

Tokens:
['The', 'Ġmoon', 'Ġshines', 'Ġover', 'Ġthe', 'Ġhills']

Token IDs:
[464, 8824, 32481, 625, 262, 18639]

Decoded text:
The moon shines over the hills


## 2.5 Understanding GPT-2's Byte-Level BPE Tokenization

GPT-2 uses byte-level Byte Pair Encoding (BPE).

Instead of requiring every possible word to exist as a separate vocabulary
entry, GPT-2 can represent words using smaller subword pieces.

Common words may be represented by one token, while uncommon or longer
words may be split into multiple tokens.

This allows GPT-2 to handle a very large variety of words while using a
fixed vocabulary of 50,257 tokens.

In [7]:
examples = [
    "moon",
    "beautiful",
    "unbelievable",
    "Kigali",
    "transformer",
    "internationalization"
]

for text in examples:
    tokens = tokenizer.tokenize(text)

    print(f"Text: {text}")
    print(f"Tokens: {tokens}")
    print(f"Number of tokens: {len(tokens)}")
    print()

Text: moon
Tokens: ['moon']
Number of tokens: 1

Text: beautiful
Tokens: ['beaut', 'iful']
Number of tokens: 2

Text: unbelievable
Tokens: ['un', 'bel', 'iev', 'able']
Number of tokens: 4

Text: Kigali
Tokens: ['K', 'ig', 'ali']
Number of tokens: 3

Text: transformer
Tokens: ['trans', 'former']
Number of tokens: 2

Text: internationalization
Tokens: ['international', 'ization']
Number of tokens: 2



## 4. Loading the Pretrained GPT-2 Model

GPT-2 is a pretrained Transformer language model.

Unlike Tiny GPT, we are not training GPT-2 from scratch here.
We will open the pretrained model and inspect its architecture,
embeddings, attention heads, and text-generation behavior.

In [6]:
# Load the pretrained GPT-2 model
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Put the model in evaluation mode
model.eval()

# Select GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move the model to the selected device
model = model.to(device)

print("Device:", device)
print("Model loaded successfully!")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device: cuda
Model loaded successfully!


In [9]:
sample = "Love is"

print("Original text:")
print(sample)

# Tiny GPT tokenization: character-level
tiny_tokens = list(sample)

# GPT-2 tokenization: BPE
gpt2_tokens = tokenizer.tokenize(sample)

print("Original:", sample)

print("\nTiny GPT")
print("Tokens:", tiny_tokens)
print("Number of tokens:", len(tiny_tokens))

print("\nGPT-2")
print("Tokens:", gpt2_tokens)
print("Number of tokens:", len(gpt2_tokens))

Original text:
Love is
Original: Love is

Tiny GPT
Tokens: ['L', 'o', 'v', 'e', ' ', 'i', 's']
Number of tokens: 7

GPT-2
Tokens: ['Love', 'Ġis']
Number of tokens: 2


### 2.8 Tiny GPT vs GPT-2 Tokenization

Both models were given the same input: "Love is".

Tiny GPT uses character-level tokenization, producing 7 tokens:
['L', 'o', 'v', 'e', ' ', 'i', 's']

GPT-2 uses Byte-Level BPE tokenization, producing only 2 tokens:
['Love', 'Ġis']

This shows that Tiny GPT represents text character by character,
while GPT-2 can represent common words or subword units as larger
tokens. Therefore, GPT-2 generally requires fewer tokens to represent
the same natural-language text.

In [10]:
!pwd

/content


---
# Task 3 — GPT-2 Architecture & Embeddings

**Owner:** Member 2

This section investigates the architecture of the pretrained GPT-2 model:
its configuration, embedding layers, transformer block components,
feed-forward sublayer, output head, parameter count, and a side-by-side
comparison with our Tiny GPT model.

## 3.1  Load the Pretrained GPT-2 Model

The model was already loaded above in Section 4.
Here we confirm it is ready and inspect its full configuration object.

In [ ]:
# Model is already loaded above as `model`
# Confirm it is in eval mode on the correct device
print("Model class  :", model.__class__.__name__)
print("Device       :", next(model.parameters()).device)
print("Training mode:", model.training, " (False = eval mode)")
print()
print("Full configuration:")
print(model.config)

## 3.2  Model Dimensions

GPT-2's key dimensions are stored in `model.config`.

| Property | Attribute | Meaning |
|---|---|---|
| Vocabulary size | `vocab_size` | Number of possible tokens |
| Context length | `n_positions` | Maximum sequence length |
| Hidden / embedding size | `n_embd` | Width of each token vector |
| Number of transformer layers | `n_layer` | Depth of the model |
| Number of attention heads | `n_head` | Parallel attention streams per layer |

In [ ]:
cfg = model.config

print("===== GPT-2 (small) Configuration =====")
print(f"Vocabulary size   : {cfg.vocab_size:,}")
print(f"Context length    : {cfg.n_positions:,} tokens")
print(f"Embedding size    : {cfg.n_embd}")
print(f"Number of layers  : {cfg.n_layer}")
print(f"Attention heads   : {cfg.n_head}")
print(f"Head dimension    : {cfg.n_embd // cfg.n_head}  (n_embd / n_head)")
print(f"FFN inner size    : {4 * cfg.n_embd}  (4 x n_embd)")
print(f"Activation        : {cfg.activation_function}")

**Recorded values for GPT-2 small:**

| Dimension | Value |
|---|---|
| Vocabulary size | 50,257 |
| Context length | 1,024 tokens |
| Hidden / embedding size | 768 |
| Number of layers | 12 |
| Number of attention heads | 12 |
| Head dimension | 64 |
| FFN inner size | 3,072 (4 x 768) |

## 3.3  Embeddings: Token Embeddings and Positional Embeddings

### Why two separate embedding tables?

A transformer processes all tokens in parallel, so it has no built-in
notion of sequence order. Two embedding tables solve different problems:

| Embedding | Purpose |
|---|---|
| **Token embedding** (`wte`) | Maps each token ID to a dense vector that captures the token's *meaning*. |
| **Positional embedding** (`wpe`) | Maps each position index (0 to 1023) to a vector that encodes *where* the token sits in the sequence. |

The two vectors are **added** together before the first transformer block,
giving every token a representation that carries both identity and position.

Without positional embeddings the model would treat
`"the cat sat"` and `"sat cat the"` identically, because attention has no
other way to distinguish positions.

In [ ]:
wte = model.transformer.wte   # token embedding table
wpe = model.transformer.wpe   # positional embedding table

print("Token embedding table (wte)      :", wte)
print()
print("Positional embedding table (wpe) :", wpe)
print()
print("Token embedding weight shape     :", wte.weight.shape,
      "  -> (vocab_size, n_embd)")
print("Positional embedding weight shape:", wpe.weight.shape,
      "  -> (n_positions, n_embd)")

In [ ]:
# Demonstrate token emb + position emb -> combined input
sample_text = "The moon shines over the hills"
token_ids_t = tokenizer.encode(sample_text, return_tensors="pt").to(device)

T = token_ids_t.size(1)

tok_embs = wte(token_ids_t)                    # (1, T, 768)
pos_ids  = torch.arange(T, device=device)      # [0, 1, ..., T-1]
pos_embs = wpe(pos_ids)                        # (T, 768)
combined = tok_embs + pos_embs                 # (1, T, 768)

print("Input text        :", sample_text)
print("Tokens            :", tokenizer.tokenize(sample_text))
print("Sequence length T :", T)
print()
print("Token embedding shape    :", tok_embs.shape, " (batch, T, n_embd)")
print("Position embedding shape :", pos_embs.shape, " (T, n_embd)")
print("Combined shape           :", combined.shape, " (batch, T, n_embd)")

## 3.4  Transformer Block Components

GPT-2 stacks 12 identical transformer blocks accessible via `model.transformer.h[i]`.

Each block applies these components in order:

```
Input
  |
  |-- LayerNorm 1 (ln_1)
  |
  |-- Masked Multi-Head Self-Attention (attn)
  |       causal mask prevents attending to future tokens
  |
  |-- Residual connection  (add original input back)
  |
  |-- LayerNorm 2 (ln_2)
  |
  |-- Feed-Forward Network (mlp)  Linear -> GELU -> Linear
  |
  |-- Residual connection  (add pre-FFN value back)
       |
     Output
```

GPT-2 uses **Pre-LayerNorm**: normalisation happens *before* each
sub-layer rather than after, which improves training stability.

In [ ]:
# Inspect the first transformer block
block = model.transformer.h[0]

print("=== Transformer Block 0 ===")
print(block)
print()
print("Sub-modules:")
for name, module in block.named_children():
    print(f"  {name:6s} -> {module.__class__.__name__}")

In [ ]:
# Inspect the attention sub-module
attn = block.attn

print("=== Masked Self-Attention ===")
print(f"Number of heads     : {cfg.n_head}")
print(f"Embedding size      : {cfg.n_embd}")
print(f"Head dimension      : {cfg.n_embd // cfg.n_head}")
print(f"c_attn (Q+K+V proj) : {attn.c_attn}")
print(f"c_proj (output proj): {attn.c_proj}")
print()
print("Causal mask (first 5x5 corner):")
# Newer transformers versions compute the mask on the fly
# We reconstruct it manually to illustrate the concept
causal_mask = torch.tril(torch.ones(5, 5, dtype=torch.int))
print(causal_mask)
print("(1 = can attend, 0 = masked out / future token)")

**Component summary for one GPT-2 block:**

| Component | Role |
|---|---|
| `ln_1` — Layer Normalisation | Stabilises activations before attention. |
| `attn` — Masked Multi-Head Self-Attention | Each token gathers information from all previous tokens. The causal mask prevents attending to future positions. |
| Residual connection | Adds the original input back after attention, preserving earlier information. |
| `ln_2` — Layer Normalisation | Stabilises activations before the FFN. |
| `mlp` — Feed-Forward Network | Applies a position-wise non-linear transformation on every token vector independently. |
| Residual connection | Adds the pre-FFN value back after the FFN. |

## 3.5  Feed-Forward Network (FFN)

After self-attention has mixed information *across* positions, the FFN
processes each token vector *independently*:

1. **Expand** — Linear projects from `n_embd=768` up to `4 x n_embd=3072`.
   This wider space gives the model room for complex transformations.

2. **Activate** — GELU introduces non-linearity so the network can learn
   which features are useful.

3. **Project** — A second Linear brings the representation back to `768`
   so the next block receives the same shape.

The FFN is where GPT-2 stores factual knowledge. Research shows that
attention builds contextual representations while the FFN acts as a
key-value memory for facts.

In [ ]:
mlp = block.mlp

print("=== Feed-Forward Network (MLP) ===")
print(mlp)
print()
print(f"Input  dimension : {cfg.n_embd}")
print(f"Hidden dimension : {4 * cfg.n_embd}  (4 x n_embd)")
print(f"Output dimension : {cfg.n_embd}")
print(f"Activation       : GELU")
print()
ffn_params = sum(p.numel() for p in mlp.parameters())
print(f"Parameters in one FFN block: {ffn_params:,}")

## 3.6  Output Head: From Hidden State to Next-Token Probabilities

After the final transformer block, two steps convert the hidden
representation into a probability distribution over the vocabulary:

```
Last block output  (batch, T, 768)
       |
  Final LayerNorm (ln_f)        -> stabilise activations
       |
  Linear projection (lm_head)   -> (batch, T, 50257) logit scores
       |
  Softmax over vocabulary        -> probability for each token
       |
  argmax / sampling              -> chosen next token
```

**Weight tying:** `lm_head` shares its weight matrix with `wte`.
This means tokens with similar meanings have consistent representations
in both input and output, and it saves ~39 M parameters.

In [ ]:
ln_f    = model.transformer.ln_f
lm_head = model.lm_head

print("Final Layer Norm (ln_f) :", ln_f)
print("Language-model head     :", lm_head)
print()
print("Weight tying check (lm_head.weight is wte.weight):",
      lm_head.weight.data_ptr() == wte.weight.data_ptr())

In [ ]:
import torch.nn.functional as F

# Forward pass -> logits -> probabilities -> top-5 predictions
prompt_text = "Roses are red, violets are"
input_ids_p = tokenizer.encode(prompt_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(input_ids_p)

logits      = outputs.logits
last_logits = logits[0, -1, :]
probs       = F.softmax(last_logits, dim=-1)
top5_probs, top5_ids = torch.topk(probs, 5)

print(f"Prompt: '{prompt_text}'")
print(f"Logits shape (full sequence): {logits.shape}")
print()
print("Top-5 next-token predictions:")
for prob, tid in zip(top5_probs, top5_ids):
    token_str = tokenizer.decode([tid.item()])
    print(f"  '{token_str}' (id={tid.item():5d}) -> {prob.item():.4f}")

## 3.7  Parameter Count

We count parameters in each part of the model to understand where
the ~124 M parameters live.

In [ ]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}  (~{total_params/1e6:.1f} M)")
print()

components = {
    "Token embeddings (wte)"     : model.transformer.wte,
    "Positional embeddings (wpe)": model.transformer.wpe,
    "Final LayerNorm (ln_f)"     : model.transformer.ln_f,
}
for name, mod in components.items():
    n = sum(p.numel() for p in mod.parameters())
    print(f"  {name:40s}: {n:>12,}")

block_params = sum(
    sum(p.numel() for p in b.parameters())
    for b in model.transformer.h
)
print(f"  {'12 Transformer blocks':40s}: {block_params:>12,}")
print()
print("  --- One transformer block breakdown ---")
for name, sub in model.transformer.h[0].named_children():
    n = sum(p.numel() for p in sub.parameters())
    print(f"    {name:10s}: {n:>10,}")

**Parameter breakdown for GPT-2 small:**

| Component | Parameters |
|---|---|
| Token embeddings (wte) | 38,597,376 |
| Positional embeddings (wpe) | 786,432 |
| 12 x transformer blocks | 85,054,464 |
| Final LayerNorm (ln_f) | 1,536 |
| **Total** | **~124,439,808** |

> `lm_head` shares weights with `wte` (weight tying), so those
> 38.6 M parameters are not double-counted.

## 3.8  Architecture Comparison: Tiny GPT vs GPT-2

Tiny GPT values are taken directly from `01_TinyGPT.ipynb`.
GPT-2 values are read live from `model.config` above.

In [ ]:
import pandas as pd

# Tiny GPT values from 01_TinyGPT.ipynb
tiny_vocab_size   = 288        # character-level vocab
tiny_context      = 256        # block_size
tiny_n_embd       = 128        # n_embd
tiny_n_layers     = 1          # single transformer block
tiny_n_heads      = 1          # single-head attention
tiny_ffn_inner    = 4 * tiny_n_embd   # 512
tiny_params       = 288_288    # from notebook output

# GPT-2 values from model.config
gpt2_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

rows = [
    ("Tokenization",            "Character-level",        "Byte-Level BPE"),
    ("Vocabulary size",         f"{tiny_vocab_size:,}",   f"{cfg.vocab_size:,}"),
    ("Context length (tokens)", f"{tiny_context:,}",      f"{cfg.n_positions:,}"),
    ("Embedding size",          f"{tiny_n_embd}",         f"{cfg.n_embd}"),
    ("Number of layers",        f"{tiny_n_layers}",       f"{cfg.n_layer}"),
    ("Attention heads",         f"{tiny_n_heads}",        f"{cfg.n_head}"),
    ("Head dimension",          f"{tiny_n_embd}",         f"{cfg.n_embd // cfg.n_head}"),
    ("FFN inner size",          f"{tiny_ffn_inner:,}",    f"{4 * cfg.n_embd:,}"),
    ("Activation function",     "GELU",                   "GELU"),
    ("Positional encoding",     "Learned",                "Learned"),
    ("LayerNorm style",         "Post-LN",                "Pre-LN"),
    ("Trainable parameters",   f"{tiny_params:,}",        f"{gpt2_params:,}"),
    ("Training data",           "Poetry corpus (custom)", "WebText (~40 GB)"),
    ("Training approach",       "From scratch",           "Pretrained"),
]

df_cmp = pd.DataFrame(rows, columns=["Property", "Tiny GPT", "GPT-2 (small)"])
df_cmp = df_cmp.set_index("Property")
print(df_cmp.to_string())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = [
    ("Vocab size",      tiny_vocab_size,     cfg.vocab_size),
    ("Context length",  tiny_context,        cfg.n_positions),
    ("Embedding size",  tiny_n_embd,         cfg.n_embd),
    ("Layers",          tiny_n_layers,       cfg.n_layer),
    ("Attn heads",      tiny_n_heads,        cfg.n_head),
    ("Params (K)",      tiny_params // 1000, gpt2_params // 1000),
]

labels    = [m[0] for m in metrics]
tiny_vals = np.array([m[1] for m in metrics], dtype=float)
gpt2_vals = np.array([m[2] for m in metrics], dtype=float)

tiny_log = np.log10(tiny_vals + 1)
gpt2_log = np.log10(gpt2_vals + 1)

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - w/2, tiny_log, w, label="Tiny GPT",    color="steelblue",  alpha=0.85)
bars2 = ax.bar(x + w/2, gpt2_log, w, label="GPT-2 small", color="darkorange", alpha=0.85)

ax.set_xlabel("Dimension", fontsize=12)
ax.set_ylabel("log10(value)", fontsize=12)
ax.set_title("Tiny GPT vs GPT-2 (small) — Architecture Dimensions (log scale)", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha="right")
ax.legend(fontsize=11)

for bar, val in zip(bars1, tiny_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f"{int(val):,}", ha="center", va="bottom", fontsize=7.5, color="steelblue")
for bar, val in zip(bars2, gpt2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f"{int(val):,}", ha="center", va="bottom", fontsize=7.5, color="darkorange")

plt.tight_layout()
plt.show()

### Key differences summary

| Dimension | Tiny GPT | GPT-2 (small) | Scale factor |
|---|---|---|---|
| Vocabulary | 288 chars | 50,257 subwords | x175 |
| Context length | 256 tokens | 1,024 tokens | x4 |
| Embedding size | 128 | 768 | x6 |
| Transformer layers | 1 | 12 | x12 |
| Attention heads | 1 | 12 | x12 |
| Parameters | ~288 K | ~124 M | x431 |

- **Tokenization:** Tiny GPT works character by character (288 tokens).
  GPT-2 uses BPE subwords (50,257 tokens), capturing richer linguistic
  units and requiring fewer tokens per sentence.
- **Depth and width:** GPT-2 is 6x wider and 12x deeper, allowing it
  to capture higher-order relationships across layers.
- **Multi-head attention:** 12 heads let GPT-2 track syntax, coreference,
  and semantics simultaneously. Tiny GPT's single head handles only one
  relationship type at a time.
- **Scale:** GPT-2 has ~431x more parameters, trained on ~40 GB of web
  text vs. our small poetry corpus.

---
## Task 3 Recap

| Sub-task | What was done |
|---|---|
| **3.1** | Confirmed model loaded; printed full config |
| **3.2** | vocab=50,257 · context=1,024 · embd=768 · layers=12 · heads=12 |
| **3.3** | Explained wte + wpe; demonstrated addition giving (batch, T, 768) |
| **3.4** | ln_1 -> attn (masked) -> residual -> ln_2 -> mlp -> residual |
| **3.5** | FFN: Linear(768->3072) -> GELU -> Linear(3072->768) per token |
| **3.6** | ln_f -> lm_head (weight-tied to wte) -> softmax -> next-token probs |
| **3.7** | ~124 M params; most in the 12 transformer blocks |
| **3.8** | Comparison table + bar chart; GPT-2 is ~431x larger than Tiny GPT |